# 08주차: PyTorch 이미지 분류 최적화

## 학습 목표
7주차에서 사용한 `ImprovedCNN`과 같은 블록 구조를 이 노트북 안에서 독립적으로 다시 정의하고, 학습률, 옵티마이저(SGD·Adam), 배치 정규화, 드롭아웃, Early Stopping이 CIFAR-10 분류 성능에 어떤 영향을 주는지 순서대로 관찰합니다. 여러 설정을 짧게 비교하는 실험과, 찾은 설정을 실제 규모로 학습하는 최종 실험을 구분해서 진행합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. GPU가 없어도 CPU에서 실행되지만 학습이 오래 걸립니다. 데이터는 7주차와 같은 방식으로 **Hugging Face Hub**에서 자동으로 내려받으므로(10초 안팎) Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다. 이 노트북은 6·7주차 파일을 불러오지 않고 데이터 다운로드, 분할, 모델, 학습 함수를 모두 새로 정의합니다.

## 관찰 질문
- 학습률이 너무 작거나 너무 크면 짧은 학습 안에서 어떤 신호로 드러날까요?
- 배치 정규화와 드롭아웃은 왜 같이 켜졌을 때 학습 안정성과 과적합 방지에 함께 도움이 될까요?
- Early Stopping은 왜 "가장 낮은 검증 손실"을 기준으로 멈추고, 마지막 에포크의 가중치를 그대로 쓰지 않을까요?


In [ ]:
# Colab에는 datasets가 기본 설치되어 있지 않을 수 있습니다. 이미 있으면 그냥 넘어갑니다.
# (로컬에서 uv로 환경을 만들었다면 이 셀은 실행하지 않아도 됩니다.)
try:
    import datasets
except ImportError:
    %pip install -q datasets


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from datasets import load_dataset
from sklearn.metrics import confusion_matrix   # 마지막에 클래스별 혼동 양상을 볼 때 쓴다

# ---------------------------------------------------------------------------
# 이 노트북의 텐서 차원 표기
#   B = 배치 크기(여기서는 128, 마지막 배치는 더 작을 수 있음)
#   이미지 텐서는 (B, 3, 32, 32), 라벨 텐서는 (B,), 모델 출력(로짓)은 (B, 10).
#   마지막 평가에서는 테스트셋 전체를 이어 붙여 (10000,)짜리 예측/정답 배열을 만든다.
# ---------------------------------------------------------------------------

def seed_everything(seed=42):
    """파이썬·numpy·PyTorch(CPU/GPU)의 난수 생성기를 모두 같은 시드로 고정한다.

    이번 주차는 "설정 하나만 바꾸고 나머지는 고정"하는 비교를 반복한다.
    비교 직전마다 이 함수를 다시 불러 초기 가중치를 똑같이 맞추는 것이 핵심이다.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# ---------------------------------------------------------------------------
# 데이터 내려받기: 7주차와 같은 방식(Hugging Face Hub)을 쓴다.
# CIFAR-10의 torchvision 기본 서버(cs.toronto.edu)는 국내·Colab에서 매우 느려
# 다운로드에만 수십 분이 걸리는 경우가 있다. HF는 CDN이라 10초 안팎이면 받아진다.
# ---------------------------------------------------------------------------

def choose_num_workers():
    """실행 환경의 CPU 개수에 맞춰 DataLoader 워커 수를 정한다.

    워커는 배치를 미리 준비해 두는 별도 프로세스다. Colab 런타임마다
    할당되는 vCPU 수가 다르므로(2개인 경우도, 8개인 경우도 있다) 고정값 대신
    실행할 때 세어서 정한다.

    - os.sched_getaffinity(0)은 "이 프로세스가 실제로 쓸 수 있는" 코어를 센다.
      Colab처럼 컨테이너로 코어를 제한하는 환경에서 os.cpu_count()는 호스트
      전체 코어를 세어 과대평가할 수 있다. 리눅스에만 있으므로 없으면 대체한다.
    - 워커를 1개만 쓰면 0개(메인 프로세스가 직접 준비)보다 오히려 느리다.
      병렬성은 없으면서 프로세스 간에 데이터를 넘기는 비용만 붙기 때문이다.
      그래서 코어가 2개 미만이면 아예 0으로 둔다.
    - 8을 넘겨도 이득이 거의 없다. 그 지점이면 이미 GPU 연산이 병목이다.
      코어 수보다 많은 워커를 만들면 오히려 느려지고 PyTorch가 경고도 낸다.
    """
    try:
        cores = len(os.sched_getaffinity(0))     # 리눅스(Colab 포함)
    except AttributeError:
        cores = os.cpu_count() or 1              # macOS, Windows 등
    return 0 if cores < 2 else min(8, cores)


def load_hf_images(repo_id, split):
    """HF Hub에서 데이터셋을 받아 (이미지 배열, 라벨 배열)로 메모리에 펼친다.

    반환값
      images: (N, 32, 32, 3) uint8 numpy 배열   labels: (N,) int64 numpy 배열

    HF는 이미지를 PNG로 압축해 저장하므로 매번 꺼내 쓰면 디코딩 때문에 느려진다.
    여기서 한 번만 전부 풀어 numpy 배열로 만들어 둔다(학습셋 146MB).
    """
    rows = load_dataset(repo_id, split=split)
    image_key = [name for name in rows.column_names if name != "label"][0]
    images = np.stack([np.asarray(image) for image in rows[image_key]])   # (N, 32, 32, 3)
    labels = np.asarray(rows["label"])                                    # (N,)
    return images, labels


class ArrayDataset(Dataset):
    """메모리의 uint8 배열을 PIL 이미지로 바꿔 transform에 넘기는 Dataset.

    한 항목은 (이미지 텐서 (3, 32, 32), 라벨 정수) 튜플이다.
    같은 배열을 transform만 바꿔 여러 번 감쌀 수 있어, 증강 있는 버전과
    없는 버전이 픽셀 데이터를 공유한다(메모리를 두 배로 쓰지 않는다).
    """

    def __init__(self, images, labels, transform):
        self.images = images            # (N, 32, 32, 3) uint8
        self.labels = labels            # (N,)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # images[index]: (32, 32, 3) uint8 -> PIL -> transform -> (3, 32, 32) 실수 텐서
        return self.transform(Image.fromarray(self.images[index])), int(self.labels[index])


# 채널별 평균·표준편차이므로 원소가 3개씩이다. R, G, B 순서.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

# base_transform: 증강 없음. 검증·테스트에 쓴다.
# ToTensor  : PIL 컬러 이미지 -> (3, 32, 32), 값 0~1
# Normalize : (3, 32, 32) -> (3, 32, 32), 모양은 그대로이고 값 범위만 바뀐다
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
# augment_transform: 7주차에서 효과를 확인한 증강. 훈련에만 쓴다.
# RandomCrop/Flip은 모양을 바꾸지 않으므로 최종 출력도 (3, 32, 32)로 같다.
augment_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# 이미지 배열은 여기서 딱 한 번만 만들고, 아래 데이터셋들이 공유한다.
train_images, train_labels = load_hf_images("uoft-cs/cifar10", "train")   # (50000, 32, 32, 3)
test_images, test_labels = load_hf_images("uoft-cs/cifar10", "test")      # (10000, 32, 32, 3)
print(f"내려받은 이미지 배열: 학습 {train_images.shape}, 테스트 {test_images.shape}, "
      f"메모리 {(train_images.nbytes + test_images.nbytes) / 1024**2:.0f}MB")

# 같은 픽셀 데이터를 transform만 다르게 해서 두 벌 만든다.
full_train_augment = ArrayDataset(train_images, train_labels, augment_transform)
full_train_base = ArrayDataset(train_images, train_labels, base_transform)
test_dataset = ArrayDataset(test_images, test_labels, base_transform)
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# 7주차와 같은 시드(42)로 같은 방식으로 나눈다. 즉 7주차와 완전히 동일한 분할이다.
# randperm(50000)은 0~49999를 섞은 (50000,) 정수 텐서. .tolist()로 파이썬 리스트로 바꾼다.
split_generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(len(full_train_base), generator=split_generator).tolist()
train_indices = permutation[:45000]     # 길이 45000 리스트
val_indices = permutation[45000:]       # 길이  5000 리스트
print("고정 분할 크기(훈련/검증):", len(train_indices), len(val_indices))

# 훈련에는 증강 버전을, 검증에는 증강 없는 버전을 쓴다. 인덱스는 서로 겹치지 않는다.
train_dataset = Subset(full_train_augment, train_indices)
val_dataset = Subset(full_train_base, val_indices)

batch_size = 128
num_workers = choose_num_workers()
print(f"사용 가능한 CPU 코어에 맞춰 num_workers={num_workers}로 설정했습니다.")
# pin_memory: GPU로 옮길 때 빨라지는 메모리 영역을 쓴다(GPU가 있을 때만 의미 있음).
loader_options = {"batch_size": batch_size, "num_workers": num_workers, "pin_memory": torch.cuda.is_available()}
# DataLoader는 낱장 (3, 32, 32)을 쌓아 (B, 3, 32, 32)로 만들어 준다.
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("전체 데이터 수(훈련/검증/테스트):", len(train_dataset), len(val_dataset), len(test_dataset))


## ImprovedCNN 구조 재정의
7주차에서 사용한 것과 같은 블록 구조(합성곱 두 번 반복 후 풀링, 채널 32→64→128)를 이 노트북 안에서 다시 정의합니다. `use_batchnorm`을 켜면 각 합성곱 뒤에 `nn.BatchNorm2d`가 추가되고, `dropout`은 분류기 앞단의 `nn.Dropout` 비율을 결정합니다. 이번 주차의 모든 실험은 이 하나의 구조에서 옵티마이저·학습률·정규화 설정만 바꾸며 진행합니다.


In [ ]:
class ImprovedCNN(nn.Module):
    """7주차와 같은 블록 구조. 이번 주차에는 두 스위치를 실제로 켜 본다.

    use_batchnorm=True : 각 합성곱 뒤에 BatchNorm2d를 넣는다.
        배치 단위로 출력을 평균 0, 표준편차 1에 가깝게 다시 맞춰 주어
        층이 깊어져도 값의 분포가 무너지지 않게 하고 학습을 안정시킨다.
        모양은 바꾸지 않는다: (B, C, H, W) -> (B, C, H, W)
    dropout=p          : 분류기 앞에서 p 비율의 값을 무작위로 0으로 만든다.
        특정 뉴런에만 의존하지 못하게 해 과적합을 줄인다. 모양은 그대로 (B, 128).
        학습할 때만 동작하고 model.eval()에서는 자동으로 꺼진다.

    전체 흐름(입력 (B, 3, 32, 32) 기준):
      (B,   3, 32, 32)
        -> block(3, 32)   -> (B,  32, 16, 16)
        -> block(32, 64)  -> (B,  64,  8,  8)
        -> block(64, 128) -> (B, 128,  4,  4)
        -> AdaptiveAvgPool2d(1) -> (B, 128, 1, 1)
        -> Flatten        -> (B, 128)
        -> Linear(128, 10)-> (B, 10)
    """
    def __init__(self, use_batchnorm=False, dropout=0.0):
        super().__init__()
        def block(in_ch, out_ch):
            """합성곱 2번 + 풀링 1번. use_batchnorm이 켜지면 각 합성곱 뒤에 BN이 붙는다.

            (B, in_ch, H, W) -> (B, out_ch, H/2, W/2)
            채널은 첫 Conv2d에서 바뀌고, H와 W는 마지막 MaxPool2d에서만 절반이 된다.
            """
            layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]   # (B, in_ch, H, W) -> (B, out_ch, H, W)
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))           # 모양 그대로
            layers += [nn.ReLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1)]   # 모양 그대로
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))           # 모양 그대로
            layers += [nn.ReLU(), nn.MaxPool2d(2)]              # 여기서 H, W가 절반
            return layers

        self.features = nn.Sequential(
            *block(3, 32),      # (B,   3, 32, 32) -> (B,  32, 16, 16)
            *block(32, 64),     # (B,  32, 16, 16) -> (B,  64,  8,  8)
            *block(64, 128),    # (B,  64,  8,  8) -> (B, 128,  4,  4)
            nn.AdaptiveAvgPool2d(1),   # (B, 128, 4, 4) -> (B, 128, 1, 1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),           # (B, 128, 1, 1) -> (B, 128)
            nn.Dropout(dropout),    # (B, 128) -> (B, 128)
            nn.Linear(128, 10),     # (B, 128) -> (B, 10)
        )

    def forward(self, x):
        # x: (B, 3, 32, 32) -> features -> (B, 128, 1, 1) -> classifier -> (B, 10)
        return self.classifier(self.features(x))

# 학습 없이 shape만 확인한다.
sample_images, sample_labels = next(iter(train_loader))    # (B, 3, 32, 32), (B,)
with torch.no_grad():
    shape_logits = ImprovedCNN(use_batchnorm=True, dropout=0.3).to(device)(sample_images.to(device))
print("입력 텐서 shape:", sample_images.shape)   # (128, 3, 32, 32)
print("출력 텐서 shape:", shape_logits.shape)     # (128, 10)


## 공통 학습·평가 함수
`train_one_epoch`, `evaluate`, `fit`은 모델·데이터로더·옵티마이저·`device`를 인자로 받는 형태로 정의합니다. 이번 주차의 모든 비교 실험이 이 함수들을 그대로 재사용하므로, 학습 절차 자체는 고정한 채 하이퍼파라미터와 데이터 조건만 바꾸어 관찰할 수 있습니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """학습 데이터를 한 바퀴 돌며 가중치를 갱신한다."""
    model.train()       # BatchNorm은 배치 통계를 갱신하고 Dropout이 켜지는 모드
    loss_sum, correct, total = 0.0, 0, 0
    for batch_images, batch_labels in loader:      # (B, 3, 32, 32), (B,)
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()                       # 이전 배치 기울기 초기화
        logits = model(batch_images)                # 순전파: (B, 3, 32, 32) -> (B, 10)
        loss = criterion(logits, batch_labels)      # (B, 10), (B,) -> () 0차원 스칼라
        loss.backward()                             # 역전파
        optimizer.step()                            # 가중치 갱신
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()   # (B, 10) -> (B,)
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    """가중치를 바꾸지 않고 손실과 정확도만 잰다."""
    model.eval()        # BatchNorm은 저장해 둔 통계를 쓰고 Dropout은 꺼지는 모드
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:  # (B, 3, 32, 32), (B,)
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)           # (B, 10)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    """epochs만큼 학습하며 기록을 쌓는다. 비교 실험들이 이 함수를 그대로 재사용한다.

    history의 각 값은 길이가 epochs인 파이썬 리스트다(텐서가 아니다).
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    """학습으로 값이 바뀌는 파라미터의 총 개수.

    numel()은 그 텐서의 원소 개수다. 예를 들어 Conv2d(3, 32, 3)의 가중치는
    (32, 3, 3, 3) 모양이므로 numel()은 32*3*3*3 = 864가 된다.
    """
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

# 이번 주차 모든 실험이 공유하는 손실 함수.
# (B, 10) 로짓과 (B,) 정수 라벨을 받아 0차원 스칼라 손실을 낸다.
criterion = nn.CrossEntropyLoss()


## 짧은 비교 실험을 위한 고정 부분집합
학습률과 옵티마이저를 비교할 때마다 45,000개 전체 훈련 데이터로 학습하면 실습 시간 안에 여러 설정을 비교하기 어렵습니다. 이 노트북은 실행마다 데이터 양을 몰래 바꾸는 숨겨진 전역 개발용 단축 스위치를 두지 않고, 처음부터 계획된 **고정 훈련 인덱스 12,000개와 고정 검증 인덱스 3,000개**만 잘라 짧은 비교 전용 데이터셋(`short_train_loader`, `short_val_loader`)을 만듭니다. 이 인덱스는 `train_indices`, `val_indices`의 앞부분을 그대로 잘라낸 것이므로 실행할 때마다 항상 같은 이미지가 선택되며, 학습률 비교와 옵티마이저 비교 모두 이 동일한 데이터셋을 사용합니다. 이후 최종 모델 학습(BatchNorm·Dropout·Early Stopping 적용)에서는 이 부분집합이 아니라 전체 훈련 데이터(`train_loader`, 45,000개)를 사용합니다.

### 관찰 질문
- 왜 "여러 설정을 짧게 반복 비교"하는 실험에는 작은 고정 부분집합이 적합하고, 최종 모델 학습에는 전체 데이터가 필요할까요?
- 부분집합 크기(12,000/3,000)가 고정되어 있지 않고 매번 랜덤하게 바뀐다면 비교 결과를 신뢰하기 어려운 이유는 무엇일까요?


In [ ]:
# 전체 train_indices/val_indices의 "앞부분만" 잘라낸다.
# 슬라이싱이라 실행할 때마다 항상 같은 이미지가 뽑힌다(무작위 재추출이 아니다).
# 이것이 중요한 이유: 학습률 비교와 옵티마이저 비교가 서로 다른 데이터로 이뤄지면
# 두 실험 결과를 나란히 놓고 이야기할 수 없다.
short_train_indices = train_indices[:12000]
short_val_indices = val_indices[:3000]
print("짧은 비교 실험용 고정 인덱스 수(훈련/검증):", len(short_train_indices), len(short_val_indices))

# 훈련은 증강 버전, 검증은 증강 없는 버전. 전체 데이터와 같은 규칙이다.
short_train_dataset = Subset(full_train_augment, short_train_indices)
short_val_dataset = Subset(full_train_base, short_val_indices)
short_train_loader = DataLoader(short_train_dataset, shuffle=True, **loader_options)
short_val_loader = DataLoader(short_val_dataset, shuffle=False, **loader_options)
print("짧은 비교 데이터 수(훈련/검증):", len(short_train_dataset), len(short_val_dataset))


## 학습률 비교
같은 초기 시드(`seed_everything(42)`)와 같은 모델 구조(`ImprovedCNN(use_batchnorm=False, dropout=0.0)`)에서 학습률만 `1e-4`, `1e-3`, `1e-2`로 바꾸어 각각 3에포크 학습합니다. 짧은 비교용 고정 데이터셋(`short_train_loader`, `short_val_loader`)을 사용하므로 조건 차이는 오직 학습률뿐입니다.

### 관찰 질문
- 학습률이 너무 작으면(`1e-4`) 손실과 검증 정확도는 3에포크 안에 얼마나 움직일까요?
- 학습률이 너무 크면(`1e-2`) 손실 곡선이 들쭉날쭉하거나 발산하는 신호가 보이나요?


In [ ]:
# 학습률(learning rate)은 기울기 방향으로 한 번에 얼마나 크게 움직일지를 정하는 값이다.
# 너무 작으면 조금씩만 움직여 느리고, 너무 크면 최적점을 지나쳐 튕겨 나간다.
learning_rates = [1e-4, 1e-3, 1e-2]
lr_histories = {}

for lr in learning_rates:
    # 매 반복마다 시드를 다시 고정한다. 그래야 세 모델이 똑같은 초기 가중치에서 출발하고,
    # 차이가 오직 학습률 때문이라고 말할 수 있다.
    seed_everything(42)
    model = ImprovedCNN(use_batchnorm=False, dropout=0.0).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    lr_histories[lr] = fit(
        model, short_train_loader, short_val_loader,   # 짧은 비교용 고정 데이터셋
        criterion, optimizer, device, epochs=3
    )


In [ ]:
# 세 학습률의 검증 곡선을 겹쳐 그린다.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lr, history in lr_histories.items():
    axes[0].plot(history["val_loss"], label=f"lr={lr}")
    axes[1].plot(history["val_acc"], label=f"lr={lr}")
axes[0].set_title("Validation Loss by Learning Rate")
axes[0].legend()
axes[1].set_title("Validation Accuracy by Learning Rate")
axes[1].legend()
plt.show()

# 1e-4: 3에포크 안에 거의 못 올라간다(보폭이 너무 작다)
# 1e-3: 가장 빠르게 오른다
# 1e-2: 곡선이 들쭉날쭉하거나 오히려 나빠진다(보폭이 커서 최적점을 지나친다)
for lr, history in lr_histories.items():
    print(f"lr={lr}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


### 학생 활동
1. 위 결과에서 어떤 학습률이 3에포크 안에 가장 빠르게 검증 정확도를 올렸는지 확인하세요.
2. `1e-2`처럼 큰 학습률에서 손실이 들쭉날쭉하거나 오히려 커지는 구간이 있다면, 왜 그런 현상이 나타나는지 적어보세요.
3. 이 결과가 "1e-3이 항상 최선"이라는 뜻은 아닙니다. 데이터셋, 모델 구조, 배치 크기가 달라지면 최적 학습률도 달라질 수 있음을 유의하세요.


## SGD와 Adam 비교
같은 초기 시드, 같은 모델 구조, 같은 짧은 비교용 데이터셋에서 `SGD(lr=0.03, momentum=0.9)`와 `Adam(lr=1e-3)`을 각각 3에포크 학습해 비교합니다.

### 관찰 질문
- 두 옵티마이저 중 3에포크 안에서 검증 정확도가 더 빨리 오르는 쪽은 어디인가요?
- 이 비교는 짧은 3에포크 실행 한 번의 결과입니다. "SGD가 항상 느리다"거나 "Adam이 항상 낫다"처럼 일반화해도 될까요?


In [ ]:
# 옵티마이저는 "기울기를 받아 가중치를 어떻게 움직일지" 정하는 규칙이다.
#   SGD(momentum=0.9): 기울기 방향으로 일정하게 움직이되, 이전에 가던 방향의
#                      관성(momentum)을 더해 진동을 줄인다. 학습률을 직접 잘 골라야 한다.
#   Adam:              파라미터마다 지금까지의 기울기 크기를 기억해 보폭을 자동 조절한다.
#                      기본 설정으로도 초반에 빠르게 수렴하는 편이다.
# lambda로 감싼 이유: 모델을 새로 만들 때마다 그 모델의 파라미터로 옵티마이저를 새로 만들어야 하기 때문이다.
optimizer_configs = {
    "SGD": lambda params: torch.optim.SGD(params, lr=0.03, momentum=0.9),
    "Adam": lambda params: torch.optim.Adam(params, lr=1e-3),
}
optimizer_histories = {}

for name, make_optimizer in optimizer_configs.items():
    seed_everything(42)     # 두 모델의 초기 가중치를 동일하게 맞춘다
    model = ImprovedCNN(use_batchnorm=False, dropout=0.0).to(device)
    optimizer = make_optimizer(model.parameters())
    optimizer_histories[name] = fit(
        model, short_train_loader, short_val_loader,
        criterion, optimizer, device, epochs=3
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in optimizer_histories.items():
    axes[0].plot(history["val_loss"], label=name)
    axes[1].plot(history["val_acc"], label=name)
axes[0].set_title("Validation Loss by Optimizer")
axes[0].legend()
axes[1].set_title("Validation Accuracy by Optimizer")
axes[1].legend()
plt.show()

# 주의: 이건 3에포크짜리 한 번의 결과다. "SGD가 항상 느리다"고 일반화하면 안 된다.
# 실제로 오래 학습하면 잘 조정된 SGD가 Adam보다 좋은 최종 성능을 내는 경우도 많다.
for name, history in optimizer_histories.items():
    print(f"{name}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


## BatchNorm·Dropout 켜기 전/후 비교
학습률·옵티마이저 비교와 같은 짧은 비교용 고정 데이터셋(`short_train_loader`, `short_val_loader`)에서, 같은 초기 시드로 `ImprovedCNN(use_batchnorm=False, dropout=0.0)`(끄기)와 `ImprovedCNN(use_batchnorm=True, dropout=0.3)`(켜기)를 각각 3에포크 학습해 검증 손실·정확도를 비교합니다. 옵티마이저는 두 경우 모두 `Adam(lr=1e-3)`으로 고정합니다.

### 관찰 질문
- 배치 정규화와 드롭아웃을 함께 켰을 때 3에포크라는 짧은 구간 안에서도 검증 손실·정확도가 다르게 움직이나요?
- 이 비교는 3에포크만 본 결과입니다. 에포크를 늘리면 두 설정의 차이가 더 커질지 작아질지 예상해 보세요.


In [ ]:
# 두 설정을 비교한다. 옵티마이저·학습률·데이터·에포크는 모두 고정하고
# 배치 정규화와 드롭아웃만 함께 켠다.
# lambda로 감싼 이유: 반복문 안에서 그때그때 새 모델을 만들기 위해서다.
bn_dropout_configs = {
    "BN off / Dropout 0.0": lambda: ImprovedCNN(use_batchnorm=False, dropout=0.0),
    "BN on / Dropout 0.3": lambda: ImprovedCNN(use_batchnorm=True, dropout=0.3),
}
bn_dropout_histories = {}

for name, make_model in bn_dropout_configs.items():
    seed_everything(42)
    model = make_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)   # 두 경우 모두 동일
    bn_dropout_histories[name] = fit(
        model, short_train_loader, short_val_loader,
        criterion, optimizer, device, epochs=3
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in bn_dropout_histories.items():
    axes[0].plot(history["val_loss"], label=name)
    axes[1].plot(history["val_acc"], label=name)
axes[0].set_title("Validation Loss: BatchNorm/Dropout Off vs On")
axes[0].legend()
axes[1].set_title("Validation Accuracy: BatchNorm/Dropout Off vs On")
axes[1].legend()
plt.show()

# 3에포크라는 짧은 구간인데도 차이가 크게 난다.
# 배치 정규화가 층마다 값의 분포를 다시 맞춰 주어 학습이 훨씬 빨리 진행되기 때문이다.
for name, history in bn_dropout_histories.items():
    print(f"{name}: 최종 검증 정확도={history['val_acc'][-1]:.2f}%, 최종 검증 손실={history['val_loss'][-1]:.4f}")


## BatchNorm, Dropout, Early Stopping
지금까지는 `use_batchnorm=False, dropout=0.0`인 `ImprovedCNN`만 사용했습니다. 이제 `use_batchnorm=True, dropout=0.3`으로 배치 정규화와 드롭아웃을 함께 켠 모델을 학습하면서, 검증 손실이 더 이상 좋아지지 않을 때 학습을 멈추고 **가장 좋았던 가중치**로 되돌리는 `EarlyStopping`을 사용합니다. `EarlyStopping`은 가중치를 파일로 저장하지 않고, `state_dict()`를 CPU 텐서로 복제해 메모리에만 보관합니다.

### 관찰 질문
- `patience=3`은 검증 손실이 몇 번 연속으로 개선되지 않아야 학습이 멈춘다는 뜻일까요?
- 왜 최적 가중치를 파일에 저장했다가 다시 읽는 대신, 메모리에 있는 `state_dict()` 복제본을 그대로 사용하는 방식을 택했을까요?


In [ ]:
class EarlyStopping:
    """검증 손실이 더 이상 좋아지지 않으면 학습을 멈추고, 가장 좋았던 가중치로 되돌린다.

    검증 손실이 최저였던 순간이 "일반화가 가장 잘 된 시점"이다. 그 뒤로도 계속
    학습하면 훈련 데이터에만 더 맞춰지므로, 마지막 에포크의 가중치를 쓰면 손해다.
    """

    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience        # 몇 번 연속으로 개선이 없으면 멈출지
        self.min_delta = min_delta      # 이 값보다 더 좋아져야 "개선"으로 인정한다
        self.best_loss = float("inf")   # 지금까지의 최저 검증 손실(스칼라)
        self.bad_epochs = 0             # 연속으로 개선되지 않은 횟수
        self.best_state = None          # 최저 손실일 때의 가중치 사본(텐서들의 딕셔너리)

    def step(self, val_loss, model):
        """매 에포크 끝에 호출한다. 이제 멈춰야 하면 True를 돌려준다."""
        if val_loss < self.best_loss - self.min_delta:
            # 개선됨: 기록을 갱신하고 카운터를 0으로 되돌린다
            self.best_loss = val_loss
            self.bad_epochs = 0
            # state_dict()는 층 이름 -> 파라미터 텐서 딕셔너리다. 예를 들어
            #   "features.0.weight" -> (32, 3, 3, 3)
            #   "features.0.bias"   -> (32,)
            #   "classifier.2.weight" -> (10, 128)
            # 각 텐서의 모양은 그대로 두고 값만 복사한다.
            # 이 텐서들은 모델 내부를 "참조"하므로 그냥 저장하면 이후 학습으로 값이 같이 바뀐다.
            # clone()으로 값을 복사해 두어야 하고, .cpu()로 옮겨 GPU 메모리를 붙잡지 않게 한다.
            self.best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            return False
        # 개선되지 않음: 카운터를 늘리고, patience에 도달했으면 멈추라고 알린다
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore(self, model):
        """기억해 둔 최적 가중치를 모델에 되돌려 놓는다.

        load_state_dict는 모양이 정확히 일치해야 통과한다. 같은 구조의 모델이므로 문제없다.
        """
        model.load_state_dict(self.best_state)


## 최종 모델 학습(전체 데이터)
최종 모델은 `ImprovedCNN(use_batchnorm=True, dropout=0.3)`을 `Adam(lr=1e-3)`으로 최대 12에포크, `EarlyStopping(patience=3)`과 함께 **전체 훈련 분할**(`train_loader`, 45,000개, 데이터 증강 포함)에서 학습합니다. 짧은 비교 실험과 달리 여기서는 지금까지 관찰한 설정(Adam, 적절한 학습률, 배치 정규화, 드롭아웃)을 실제 규모의 데이터에 적용합니다.


In [ ]:
# 최종 모델: 지금까지 관찰한 좋은 설정을 모두 적용하고 전체 훈련 데이터로 학습한다.
#   - Adam(lr=1e-3): 학습률 비교와 옵티마이저 비교에서 가장 좋았던 조합
#   - BatchNorm + Dropout: 짧은 비교에서 뚜렷한 개선을 보인 조합
#   - train_loader: 짧은 비교용 12,000장이 아니라 전체 45,000장(증강 포함)
seed_everything(42)
final_model = ImprovedCNN(use_batchnorm=True, dropout=0.3).to(device)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)

# patience=3: 검증 손실이 3번 연속 개선되지 않으면 멈춘다.
early_stopping = EarlyStopping(patience=3, min_delta=0.0)
max_epochs = 12     # 최대치일 뿐, 조기 종료되면 더 일찍 끝난다
final_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

# fit()을 쓰지 않고 직접 루프를 도는 이유: 매 에포크 끝에 early_stopping.step()을
# 호출해 멈출지 판단해야 하기 때문이다.
for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(final_model, train_loader, criterion, final_optimizer, device)
    val_loss, val_accuracy = evaluate(final_model, val_loader, criterion, device)
    for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
        final_history[key].append(value)
    print(f"Epoch {epoch}/{max_epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%, val_loss={val_loss:.4f}")

    # 정확도가 아니라 검증 "손실"을 기준으로 삼는다.
    # 손실은 예측 확신도까지 반영하므로 정확도보다 먼저 나빠지는 신호를 준다.
    stop_now = early_stopping.step(val_loss, final_model)
    if stop_now:
        print(f"검증 손실이 {early_stopping.patience}번 연속 개선되지 않아 {epoch} 에포크에서 조기 종료합니다.")
        break

# 중요: 마지막 에포크의 가중치가 아니라, 검증 손실이 가장 낮았던 시점의 가중치로 되돌린다.
# 끝까지 12에포크를 돌았더라도 최저 손실이 중간 에포크였다면 그쪽 가중치를 쓴다.
early_stopping.restore(final_model)
plot_history(final_history, "Final ImprovedCNN (BatchNorm+Dropout)")


## 최종 평가: 테스트 정확도와 혼동 행렬
Early Stopping이 메모리에서 복원한 최적 가중치로 테스트 정확도를 계산하고, `sklearn.metrics.confusion_matrix`로 클래스 간 혼동 양상을 확인합니다. 대각선이 아닌 칸의 값이 크다면 두 클래스가 자주 헷갈린다는 뜻입니다. 클래스별 정확도 막대그래프로 어떤 클래스가 특히 어려운지도 살펴봅니다.


In [ ]:
# restore()로 되돌린 최적 가중치로 테스트 성능을 잰다.
final_test_loss, final_test_accuracy = evaluate(final_model, test_loader, criterion, device)
print(f"최종 모델 테스트 손실: {final_test_loss:.4f}, 테스트 정확도: {final_test_accuracy:.2f}%")

# --- 테스트셋 전체의 예측과 정답을 모은다 ---
final_model.eval()
all_predictions, all_labels = [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:          # (B, 3, 32, 32), (B,)
        logits = final_model(batch_images.to(device))       # (B, 10)
        all_predictions.append(logits.argmax(dim=1).cpu())  # (B,) 짜리 텐서를 리스트에 쌓는다
        all_labels.append(batch_labels)                     # (B,)
# torch.cat은 (B,) 텐서 79개를 이어 붙여 (10000,) 하나로 만든다.
# .numpy()로 sklearn이 요구하는 numpy 배열로 바꾼다. 모양은 그대로 (10000,).
all_predictions = torch.cat(all_predictions).numpy()   # (10000,)
all_labels = torch.cat(all_labels).numpy()             # (10000,)

# --- 혼동 행렬 ---
# (10000,) 정답과 (10000,) 예측을 받아 (10, 10) 행렬을 만든다.
# confusion[i][j] = "실제로 i인데 j로 예측한 개수"
# 대각선(i == j)은 맞힌 개수, 대각선 밖은 틀린 개수다. 전체 합은 10000이다.
confusion = confusion_matrix(all_labels, all_predictions)   # (10, 10)
fig, axis = plt.subplots(figsize=(7, 6))
image_handle = axis.imshow(confusion, cmap="Blues")   # (10, 10)을 이미지처럼 그린다. 값이 클수록 진한 파랑
axis.set_xticks(range(10))
axis.set_xticklabels(class_names, rotation=45, ha="right")
axis.set_yticks(range(10))
axis.set_yticklabels(class_names)
axis.set_xlabel("Predicted class")
axis.set_ylabel("Actual class")
axis.set_title("Confusion Matrix")
plt.colorbar(image_handle, ax=axis)
plt.tight_layout()
plt.show()

# --- 클래스별 정확도 ---
# diagonal()  : (10, 10) -> (10,)  각 클래스에서 맞힌 개수
# sum(axis=1) : (10, 10) -> (10,)  행 방향 합 = 그 클래스의 실제 이미지 개수(각 1000장)
# 둘을 나누면 (10,) 모양의 클래스별 정확도가 된다.
per_class_accuracy = confusion.diagonal() / confusion.sum(axis=1)   # (10,)
fig, axis = plt.subplots(figsize=(9, 4))
axis.bar(class_names, per_class_accuracy)
axis.set_ylabel("Accuracy")
axis.set_title("Test Accuracy by Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 어떤 클래스가 특히 어려운지 확인해 보자. 보통 cat/dog, bird/deer처럼
# 모양과 배경이 비슷한 동물 클래스가 낮게 나온다.
for name, accuracy in zip(class_names, per_class_accuracy):
    print(f"{name}: {accuracy:.2%}")


## 미니 챌린지와 정리

### 학생 활동
1. 학습률 `3e-4`와 `1e-3` 중 하나를 선택하세요.
2. Dropout `0.2`와 `0.4` 중 하나를 선택하세요.
3. 오늘 관찰한 학습률 비교, BatchNorm·Dropout 효과를 근거로 왜 그 조합을 선택했는지 2~3문장으로 설명하세요.
4. 선택한 조합 **하나만** 아래 코드의 `chosen_lr`, `chosen_dropout`에 넣어 학습하고 오늘의 최종 모델과 비교하세요. 네 가지 조합을 모두 자동으로 돌리지 않습니다. 그렇게 하면 실습 시간 안에 결과를 확인하기 어렵습니다.

오늘 다룬 학습률, 옵티마이저, 배치 정규화, 드롭아웃, Early Stopping은 모두 "하이퍼파라미터 하나를 바꾸고 나머지는 고정해 관찰"하는 절차였습니다. 새로운 데이터셋을 만나도 같은 절차를 그대로 적용해 볼 수 있습니다.


In [ ]:
# 여기 두 값만 바꿔서 딱 한 번 학습해 보세요. 네 조합을 모두 돌리지 않습니다.
chosen_lr = 3e-4        # 3e-4 또는 1e-3 중 선택
chosen_dropout = 0.2    # 0.2 또는 0.4 중 선택

# 최종 모델과 나머지 조건(구조, BatchNorm, 옵티마이저 종류, 데이터, 최대 에포크)은 모두 같다.
# 그래야 차이가 내가 바꾼 두 값 때문이라고 말할 수 있다.
seed_everything(42)
challenge_model = ImprovedCNN(use_batchnorm=True, dropout=chosen_dropout).to(device)
challenge_optimizer = torch.optim.Adam(challenge_model.parameters(), lr=chosen_lr)
challenge_early_stopping = EarlyStopping(patience=3, min_delta=0.0)
challenge_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(challenge_model, train_loader, criterion, challenge_optimizer, device)
    val_loss, val_accuracy = evaluate(challenge_model, val_loader, criterion, device)
    for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
        challenge_history[key].append(value)
    print(f"Epoch {epoch}/{max_epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%")
    if challenge_early_stopping.step(val_loss, challenge_model):
        print(f"{epoch} 에포크에서 조기 종료합니다.")
        break

challenge_early_stopping.restore(challenge_model)
challenge_test_loss, challenge_test_accuracy = evaluate(challenge_model, test_loader, criterion, device)
print(f"챌린지 모델(lr={chosen_lr}, dropout={chosen_dropout}) 테스트 정확도: {challenge_test_accuracy:.2f}%")
print(f"오늘의 최종 모델 테스트 정확도: {final_test_accuracy:.2f}%")
# 최종 모델보다 낮게 나와도 실패가 아니다. 왜 그런 차이가 났는지 설명할 수 있으면 된다.
# 예를 들어 lr=3e-4는 1e-3보다 보폭이 작아 같은 에포크 안에서는 덜 수렴한다.
